In [1]:
# 1. Imports

from pathlib import Path
import time


# ============================================================
# EXTERNE BIBLIOTHEKEN - Datenverarbeitung & Analyse
# ============================================================
import numpy as np                           # Numerische Berechnungen
import pandas as pd                          # Datenanalyse und TimeSeriesManagement
import ccxt                                  # Kryptobörsen-API (Binance etc.)

from dataclasses import dataclass
import configparser
import hmac
import hashlib
import requests
import pyarrow

In [13]:
# 2. Configuration

# Change run date
RUN_DATE = "2026-03-13"
TOP_N = 100
VS_CURRENCY = "usd"


PROJECT_ROOT = Path.cwd().parent.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
FETCHED_DIR = RAW_DIR / "fetched"
PROCESSED_DIR = DATA_DIR / "processed"

ASSETS_PATH = FETCHED_DIR / "assets.parquet"
ASSET_DAILY_PATH = FETCHED_DIR / "asset_daily"  
ASSET_OHLCV_5M_DIR = FETCHED_DIR / "asset_ohlcv_5m"

SRC_PATH = PROJECT_ROOT / "Src"
SECRET_PATH = SRC_PATH / "secret.cfg"

TARGET_VOLUME_EXCHANGE = "binance"
USD_LIKE_QUOTES = ["USDT", "USDC", "USD"]

OHLCV_INTERVAL = "5m"
BINANCE_KLINE_LIMIT = 1000
OHLCV_START_DATE = RUN_DATE
OHLCV_END_DATE_EXCLUSIVE = "2026-03-14"

def controlpath():
    print(f"Projektverzeichnis:            {PROJECT_ROOT}")
    print(f"Datenverzeichnis:              {DATA_DIR}")
    print(f"Rohdatenverzeichnis:           {RAW_DIR}")
    print(f"Gefetchte Daten:               {FETCHED_DIR}")
    print(f"Verarbeitete Daten:            {PROCESSED_DIR}")
    print(f"Source-Code:                   {SRC_PATH}")
    print(f"Secret:                        {SECRET_PATH}")
    print()

    checks = {
        "Assets-Datei":              ASSETS_PATH,
        "Asset-Daily-Verzeichnis":   ASSET_DAILY_PATH,
        "Asset-OHLCV-5M-Verzeichnis": ASSET_OHLCV_5M_DIR,
    }

    for name, path in checks.items():
        exists = path.exists()
        symbol = "✅" if exists else "❌ FEHLT"
        print(f"  {symbol}  {name}: {path}")

        # Falls Ordner → Inhalt anzeigen
        if exists and path.is_dir():
            files = sorted(path.glob("*.parquet"))
            if files:
                for f in files:
                    print(f"       📄 {f.name}")
            else:
                print(f"       ⚠️  Ordner ist leer — keine .parquet Dateien!")


OHLCV_5M_COLUMNS = [
    "timestamp",
    "asset_id",
    "exchange",
    "market_symbol",
    "open",
    "high",
    "low",
    "close",
    "volume_base",
    "volume_quote",
    "trade_count",
]

controlpath()

Projektverzeichnis:            c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit
Datenverzeichnis:              c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data
Rohdatenverzeichnis:           c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data\raw
Gefetchte Daten:               c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data\raw\fetched
Verarbeitete Daten:            c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data\processed
Source-Code:                   c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\Src
Secret:                        c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\Src\secret.cfg

  ✅  Assets-Datei: c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data\raw\fetched\assets.parquet
  ✅  Asset-Daily-Verzeichnis: c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data\raw\fetched\asset_daily
       📄 asset_daily.parquet
  ✅  Asset-

In [16]:
# =========================
# Konfiguration
# =========================
BASE_URL               = "https://data-api.binance.vision"
INTERVAL               = "5m"
SLEEP_BETWEEN_REQUESTS = 0.15

# =========================
# Public REST
# =========================
SESSION = requests.Session()

def public_get(path: str, params: dict | None = None):
    """
    Öffentlicher GET-Request an Binance API.
    Kein API-Key nötig — /api/v3/klines ist ein Public Endpoint.
    """
    url = f"{BASE_URL}{path}"
    r = SESSION.get(url, params=params, timeout=20)
    if r.status_code >= 400:
        raise RuntimeError(f"Binance Fehler {r.status_code}: {r.text}")
    return r.json()


# =========================
# Parquet Speicherung
# =========================
def json_to_parquet(data: list[dict]) -> None:
    """
    Speichert OHLCV-Daten als Parquet — pro Monat eine Datei.
    - Datei existiert bereits → zusammenführen + deduplizieren
    - Datei existiert nicht   → neu erstellen
    Sortierung: market_symbol → timestamp
    """
    ASSET_OHLCV_5M_DIR.mkdir(parents=True, exist_ok=True)

    df = pd.DataFrame(data)
    df["timestamp"] = df["timestamp"].dt.tz_convert("UTC")
    df["_year"]     = df["timestamp"].dt.year
    df["_month"]    = df["timestamp"].dt.month

    for (year, month), month_df in df.groupby(["_year", "_month"]):
        month_df = month_df.drop(columns=["_year", "_month"])
        out      = ASSET_OHLCV_5M_DIR / f"ohlcv_{year:04d}-{month:02d}.parquet"

        if out.exists():
            df_all = (
                pd.concat([pd.read_parquet(out), month_df], ignore_index=True)
                .drop_duplicates(subset=["timestamp", "market_symbol"])
                .sort_values(["market_symbol", "timestamp"])
                .reset_index(drop=True)
            )
            print(f"  [PARQUET] Aktualisiert: {out.name} ({len(df_all)} Zeilen)")
        else:
            df_all = (
                month_df
                .sort_values(["market_symbol", "timestamp"])
                .reset_index(drop=True)
            )
            print(f"  [PARQUET] Erstellt: {out.name} ({len(df_all)} Zeilen)")

        df_all.to_parquet(out, index=False)


# =========================
# Ticker-Liste laden
# =========================
def load_ticker_list() -> pd.DataFrame:
    """
    Lädt die bereits von 01_assets.ipynb gefilterten und
    gespeicherten Assets.

    01_assets.ipynb hat bereits folgende Filter angewendet:
      ✅ Top 100 Market Cap
      ✅ 24h Volumen > $50 Mio
      ✅ Gelistet auf mind. 10 Major Exchanges
      ✅ Auf Binance handelbar (binance_volume_24h_usd > 0)

    → Kein erneutes Filtern nötig!

    Rückgabe: DataFrame mit asset_id, symbol, market_symbol
    """
    # ── assets.parquet laden → enthält symbol ──
    df_assets = pd.read_parquet(ASSETS_PATH)
    print(f"[TICKER] Assets geladen: {len(df_assets)} Einträge")

    # ── asset_daily laden → neuesten Tag nehmen ──
    parquet_files = sorted(ASSET_DAILY_PATH.glob("*.parquet"))
    print(f"[TICKER] Gefundene Daily-Dateien: {[f.name for f in parquet_files]}")

    df_daily = pd.concat(
        [pd.read_parquet(f) for f in parquet_files],
        ignore_index=True
    )

    # Neuesten Tag nehmen
    latest_date = df_daily["date"].max()
    df_latest   = df_daily[df_daily["date"] == latest_date].copy()
    print(f"[TICKER] Neuester Tag: {latest_date} ({len(df_latest)} Assets)")

    # ── Join: asset_id → symbol ──
    df = df_latest.merge(
        df_assets[["asset_id", "symbol", "name"]],
        on="asset_id",
        how="left"
    )

    # ── market_symbol bauen: SYMBOL + USDT ──
    # z.B. "BTC" → "BTCUSDT"
    # Nur Assets mit Binance-Volumen → sind auf Binance handelbar
    df = df[df["binance_volume_24h_usd"].notna()]
    df = df[df["binance_volume_24h_usd"] > 0]
    df["market_symbol"] = df["symbol"].str.upper() + "USDT"

    result = df[[
        "asset_id", "symbol", "name", "market_symbol",
        "market_cap_usd", "global_volume_24h_usd", "binance_volume_24h_usd"
    ]].reset_index(drop=True)

    print(f"\n[TICKER] Ticker-Liste ({len(result)} Assets):")
    print(result[["symbol", "market_symbol", "market_cap_usd"]].to_string())

    return result


# =========================
# Hauptfunktion
# =========================
def fetch_ohlcv_for_month() -> None:
    """
    Lädt historische 5m OHLCV-Daten von Binance für alle
    Assets aus asset_daily.parquet und speichert pro Monat
    eine Parquet-Datei.

    Voraussetzung: 01_assets.ipynb wurde bereits ausgeführt.
    Filter wurden dort bereits angewendet — hier kein erneutes Filtern.
    """
    print("=" * 60)
    print("Starte OHLCV-Datenbeschaffung")
    print("=" * 60)

    # ── Schritt 1: Ticker aus gespeicherten Dateien laden ──
    df_tickers = load_ticker_list()
    ticker_map = df_tickers.set_index("asset_id")["market_symbol"].to_dict()
    # ticker_map = {1: "BTCUSDT", 2: "ETHUSDT", ...}

    # ── Schritt 2: Asset-Daily Dateien durchgehen ──
    parquet_files = sorted(ASSET_DAILY_PATH.glob("*.parquet"))
    print(f"\n[FETCH] Monatsdateien: {[f.name for f in parquet_files]}")

    for file in parquet_files:
        print(f"\n[FETCH] Verarbeite: {file.name}")
        df_month         = pd.read_parquet(file)
        df_month["date"] = pd.to_datetime(df_month["date"]).dt.date

        # Nur Assets behalten die in ticker_map sind
        df_month = df_month[df_month["asset_id"].isin(ticker_map)]
        print(f"[FETCH] {len(df_month)} Zeilen")

        # ── Schritt 3: Pro Tag iterieren ──
        for current_date, df_day in df_month.groupby("date"):
            asset_ids = df_day["asset_id"].unique().tolist()
            symbols   = [ticker_map[aid] for aid in asset_ids if aid in ticker_map]
            print(f"\n[FETCH] {current_date}: {len(symbols)} Ticker")

            all_rows = []

            for market_symbol in symbols:
                try:
                    raw = public_get("/api/v3/klines", params={
                        "symbol":    market_symbol,
                        "interval":  INTERVAL,
                        "startTime": int(pd.Timestamp(current_date, tz="UTC").timestamp() * 1000),
                        "endTime":   int((pd.Timestamp(current_date, tz="UTC") + pd.Timedelta(days=1)).timestamp() * 1000),
                        "limit":     1000,  # 5m = max 288 Kerzen pro Tag
                    })

                    for row in raw:
                        all_rows.append({
                            "timestamp":     pd.to_datetime(int(row[0]), unit="ms", utc=True),
                            "market_symbol": market_symbol,
                            "open":          float(row[1]),
                            "high":          float(row[2]),
                            "low":           float(row[3]),
                            "close":         float(row[4]),
                            "volume_base":   float(row[5]),
                            "volume_quote":  float(row[7]),
                            "trade_count":   int(row[8]),
                        })

                    print(f"  [OK] {market_symbol}: {len(raw)} Kerzen")
                    time.sleep(SLEEP_BETWEEN_REQUESTS)

                except Exception as e:
                    print(f"  [FEHLER] {market_symbol}: {e}")

            # ── Schritt 4: Speichern ──
            if all_rows:
                print(f"  [SAVE] {len(all_rows)} Zeilen → Parquet")
                json_to_parquet(all_rows)
            else:
                print(f"  [SKIP] Keine Daten für {current_date}")

    print("\n" + "=" * 60)
    print("OHLCV-Datenbeschaffung abgeschlossen")
    print("=" * 60)


# =========================
# Ausführen
# =========================
fetch_ohlcv_for_month()

Starte OHLCV-Datenbeschaffung
[TICKER] Assets geladen: 100 Einträge
[TICKER] Gefundene Daily-Dateien: ['asset_daily.parquet']
[TICKER] Neuester Tag: 2026-03-13 (100 Assets)

[TICKER] Ticker-Liste (60 Assets):
     symbol market_symbol  market_cap_usd
0       BTC       BTCUSDT   1489151736234
1       ETH       ETHUSDT    284910320289
2      USDT      USDTUSDT    184033487448
3       XRP       XRPUSDT     93848427958
4       BNB       BNBUSDT     92814845420
5      USDC      USDCUSDT     79307642306
6       SOL       SOLUSDT     55094474070
7       TRX       TRXUSDT     28020678539
8      DOGE      DOGEUSDT     15685601122
9       ADA       ADAUSDT     10626275608
10      BCH       BCHUSDT      9641230635
11     LINK      LINKUSDT      7083258709
12     USDE      USDEUSDT      5922034835
13      XLM       XLMUSDT      5777615067
14      ZEC       ZECUSDT      4673427377
15     USD1      USD1USDT      4545402134
16      LTC       LTCUSDT      4530602409
17     AVAX      AVAXUSDT      4490

,asset_id,symbol,market_symbol
0,1,BTC,BTC/USDT
1,2,ETH,ETH/USDT
2,3,USDT,USDT/USD
3,4,XRP,XRP/USDT
4,5,BNB,BNB/USDT
...,...,...,...
95,96,FET,FET/USDT
96,97,VIRTUAL,VIRTUAL/USDT
97,98,PENGU,PENGU/USDT
98,99,CAKE,CAKE/USDT


Fetched 289 rows for BTC (BTC/USDT)
Fetched 289 rows for ETH (ETH/USDT)
Fetched 289 rows for USDT (USDT/USD)
Fetched 289 rows for XRP (XRP/USDT)
Fetched 289 rows for BNB (BNB/USDT)
Fetched 289 rows for USDC (USDC/USDT)
Fetched 289 rows for SOL (SOL/USDT)
Fetched 289 rows for TRX (TRX/USDT)
OHLCV fetch failed for FIGR_HELOC (nan): can only concatenate str (not "float") to str
Fetched 289 rows for DOGE (DOGE/USDT)
OHLCV fetch failed for WBT (nan): can only concatenate str (not "float") to str
OHLCV fetch failed for USDS (nan): can only concatenate str (not "float") to str
Fetched 289 rows for ADA (ADA/USDT)
OHLCV fetch failed for HYPE (nan): can only concatenate str (not "float") to str
Fetched 289 rows for BCH (BCH/USDT)
OHLCV fetch failed for LEO (nan): can only concatenate str (not "float") to str
Fetched 289 rows for LINK (LINK/USDT)
OHLCV fetch failed for XMR (nan): can only concatenate str (not "float") to str
Fetched 289 rows for USDE (USDE/USDT)
OHLCV fetch failed for CC (nan): c

,timestamp,asset_id,exchange,market_symbol,open,high,low,close,volume_base,volume_quote,trade_count
0,2026-03-13 00:00:00+00:00,1,binance,BTC/USDT,70541.3400,70541.3400,70386.0100,70480.5400,177.17450,1.249181e+07,16395
1,2026-03-13 00:05:00+00:00,1,binance,BTC/USDT,70480.5400,70633.7900,70480.5400,70633.7800,47.60617,3.360137e+06,16210
2,2026-03-13 00:10:00+00:00,1,binance,BTC/USDT,70633.7900,71118.5900,70633.7800,71118.5800,438.21589,3.108579e+07,45915
3,2026-03-13 00:15:00+00:00,1,binance,BTC/USDT,71118.5800,71680.0000,71105.3300,71270.1800,594.63126,4.246651e+07,76327
4,2026-03-13 00:20:00+00:00,1,binance,BTC/USDT,71270.1800,71759.6000,71260.4200,71530.0000,367.08836,2.628977e+07,63933
...,...,...,...,...,...,...,...,...,...,...,...
17335,2026-03-13 23:40:00+00:00,100,binance,TUSD/USDT,0.9994,0.9994,0.9994,0.9994,1557.00000,1.556066e+03,16
17336,2026-03-13 23:45:00+00:00,100,binance,TUSD/USDT,0.9995,0.9995,0.9994,0.9994,69.00000,6.895980e+01,7
17337,2026-03-13 23:50:00+00:00,100,binance,TUSD/USDT,0.9994,0.9995,0.9994,0.9995,152.00000,1.519197e+02,7
17338,2026-03-13 23:55:00+00:00,100,binance,TUSD/USDT,0.9994,0.9994,0.9994,0.9994,40.00000,3.997600e+01,5


,asset_id,symbol,market_symbol,error_message
0,9,FIGR_HELOC,NaN,"TypeError('can only concatenate str (not ""floa..."
1,11,WBT,NaN,"TypeError('can only concatenate str (not ""floa..."
2,12,USDS,NaN,"TypeError('can only concatenate str (not ""floa..."
3,14,HYPE,NaN,"TypeError('can only concatenate str (not ""floa..."
4,16,LEO,NaN,"TypeError('can only concatenate str (not ""floa..."
5,18,XMR,NaN,"TypeError('can only concatenate str (not ""floa..."
6,20,CC,NaN,"TypeError('can only concatenate str (not ""floa..."
7,27,RAIN,NaN,"TypeError('can only concatenate str (not ""floa..."
8,28,DAI,NaN,"TypeError('can only concatenate str (not ""floa..."
9,30,PYUSD,NaN,"TypeError('can only concatenate str (not ""floa..."


asset_ohlcv_5m validation checks passed.


Saved monthly OHLCV 5m files:
../../data/raw/fetched/asset_ohlcv_5m/asset_ohlcv_5m_2026-03.parquet


OHLCV 5m summary
OHLCV_START_DATE: 2026-03-13
OHLCV_END_DATE_EXCLUSIVE: 2026-03-14
rows: 17340
assets with errors: 40
OHLCV directory: ../../data/raw/fetched/asset_ohlcv_5m


,timestamp,asset_id,exchange,market_symbol,open,high,low,close,volume_base,volume_quote,trade_count,year,month
0,2026-03-13 00:00:00+00:00,1,binance,BTC/USDT,70541.3400,70541.3400,70386.0100,70480.5400,177.17450,1.249181e+07,16395,2026,3
1,2026-03-13 00:05:00+00:00,1,binance,BTC/USDT,70480.5400,70633.7900,70480.5400,70633.7800,47.60617,3.360137e+06,16210,2026,3
2,2026-03-13 00:10:00+00:00,1,binance,BTC/USDT,70633.7900,71118.5900,70633.7800,71118.5800,438.21589,3.108579e+07,45915,2026,3
3,2026-03-13 00:15:00+00:00,1,binance,BTC/USDT,71118.5800,71680.0000,71105.3300,71270.1800,594.63126,4.246651e+07,76327,2026,3
4,2026-03-13 00:20:00+00:00,1,binance,BTC/USDT,71270.1800,71759.6000,71260.4200,71530.0000,367.08836,2.628977e+07,63933,2026,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
17335,2026-03-13 23:40:00+00:00,100,binance,TUSD/USDT,0.9994,0.9994,0.9994,0.9994,1557.00000,1.556066e+03,16,2026,3
17336,2026-03-13 23:45:00+00:00,100,binance,TUSD/USDT,0.9995,0.9995,0.9994,0.9994,69.00000,6.895980e+01,7,2026,3
17337,2026-03-13 23:50:00+00:00,100,binance,TUSD/USDT,0.9994,0.9995,0.9994,0.9995,152.00000,1.519197e+02,7,2026,3
17338,2026-03-13 23:55:00+00:00,100,binance,TUSD/USDT,0.9994,0.9994,0.9994,0.9994,40.00000,3.997600e+01,5,2026,3


,asset_id,symbol,market_symbol,error_message
0,9,FIGR_HELOC,NaN,"TypeError('can only concatenate str (not ""floa..."
1,11,WBT,NaN,"TypeError('can only concatenate str (not ""floa..."
2,12,USDS,NaN,"TypeError('can only concatenate str (not ""floa..."
3,14,HYPE,NaN,"TypeError('can only concatenate str (not ""floa..."
4,16,LEO,NaN,"TypeError('can only concatenate str (not ""floa..."
5,18,XMR,NaN,"TypeError('can only concatenate str (not ""floa..."
6,20,CC,NaN,"TypeError('can only concatenate str (not ""floa..."
7,27,RAIN,NaN,"TypeError('can only concatenate str (not ""floa..."
8,28,DAI,NaN,"TypeError('can only concatenate str (not ""floa..."
9,30,PYUSD,NaN,"TypeError('can only concatenate str (not ""floa..."
